# Foundry Image Evaluation Demo

Run an image-caption evaluation in an East US 2 Microsoft Foundry project with the Foundry SDK 2.x, Microsoft Entra authentication, a Foundry model target, and a cloud evaluation run.

## Prerequisites

- Set `FOUNDRY_PROJECT_ENDPOINT` to an East US 2 project endpoint.
- The project must contain a vision-capable `gpt-4o-mini` deployment (override with `FOUNDRY_IMAGE_MODEL`).
- Authenticate with a credential supported by `DefaultAzureCredential`, such as `az login`.
- Install `requirements.txt` in the notebook environment.

## SDK boundary

The notebooks use `AIProjectClient` with `DefaultAzureCredential` for project access and client creation. `azure-ai-projects` 2.7.0 exposes stable `TargetCompletionEvalRunDataSource`, `AzureAIModelTargetParam`, and `ModelSamplingConfigParam` types, which are used for each cloud run.

The current Foundry evaluation create operation is still exposed only through the OpenAI-compatible protocol client returned by `AIProjectClient.get_openai_client()`. The 2.7.0 package does not expose stable Azure types for the custom eval schema, score-model grader, or inline source/input-message members (the stable data-source annotations themselves still reference `openai.types`). Those small payloads are therefore isolated as plain dictionaries in this notebook/helper, with no direct `openai` import, direct client construction, API key, raw route, or account endpoint.


In [1]:
import os
from dotenv import load_dotenv
from scripts.eval_utils import FoundryEvaluationClient, build_model_target_data_source

load_dotenv()
PROJECT_ENDPOINT = os.environ["FOUNDRY_PROJECT_ENDPOINT"]
if "eastus2" not in PROJECT_ENDPOINT.lower():
    raise ValueError("These demos must run against an East US 2 Foundry project endpoint.")

IMAGE_MODEL = os.getenv("FOUNDRY_IMAGE_MODEL", "gpt-4o-mini")
print("Foundry SDK client configuration ready for East US 2.")

Foundry SDK client configuration ready for East US 2.


## Prepare inline image evaluation data

The helper writes ten deterministic public image URLs with literal reference captions.

In [2]:
from scripts.image_utils import load_and_create_image_dataset, display_items

load_and_create_image_dataset("picsum", max_samples=10)
display_items(3)

C:\Users\prakharg\.copilot\session-state\f1833604-4ea0-4404-893e-2421eb3ad912\files\evaluation-venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


✅ Created evaluation file with 10 items
{
  "item": {
    "image_url": "https://fastly.picsum.photos/id/10/512/512.jpg?hmac=Xno3CU-pY4UWutaA1XRUThqgbV_vn7kN5PjrlkRw-9Q",
    "caption": "A view of a calm ocean with distant islands, framed by lush green trees."
  }
}
{
  "item": {
    "image_url": "https://fastly.picsum.photos/id/20/512/512.jpg?hmac=pGfehaSsk_8XvaGx8DJNw0BHdYqBj6CZWPFu6sTNPiA",
    "caption": "A workspace featuring a laptop, smartphone, books, and design materials."
  }
}
{
  "item": {
    "image_url": "https://fastly.picsum.photos/id/30/512/512.jpg?hmac=ph0RnBNUKGfVgtH1-jDWPbN0TZXOXat6xKzPazurl_Y",
    "caption": "A white mug featuring a design inspired by a Cuban stamp."
  }
}


In [3]:
import json

with open("data/image_emotion_evaluation.jsonl", encoding="utf-8") as f:
    eval_items = [json.loads(line) for line in f]
print(f"Prepared {len(eval_items)} image evaluation items.")

Prepared 10 image evaluation items.


## Create and run the Foundry cloud evaluation

The Foundry model target generates a caption from each image. A score-model grader judges the generated caption against the reference caption and image.

In [4]:
data_source_config = {
    "type": "custom",
    "item_schema": {
        "type": "object",
        "properties": {
            "image_url": {"type": "string", "description": "Public image URL."},
            "caption": {"type": "string", "description": "Reference caption."},
        },
        "required": ["image_url", "caption"],
    },
    "include_sample_schema": True,
}

score_model_grader = {
    "type": "score_model",
    "name": "image_caption_match",
    "model": IMAGE_MODEL,
    "input": [
        {
            "role": "system",
            "content": "Judge how accurately {{sample.output_text}} describes the image and matches {{item.caption}}. Return only a float from 0.0 to 1.0.",
        },
        {
            "role": "user",
            "content": [
                {"type": "input_text", "text": "Reference caption: {{item.caption}}"},
                {"type": "input_image", "image_url": "{{item.image_url}}", "detail": "auto"},
            ],
        },
    ],
    "range": [0.0, 1.0],
    "pass_threshold": 0.5,
}

input_messages = {
    "type": "template",
    "template": [
        {
            "role": "system",
            "content": "Describe the supplied image accurately and concisely.",
        },
        {
            "role": "user",
            "type": "message",
            "content": {"type": "input_text", "text": "Write one short caption."},
        },
        {
            "role": "user",
            "type": "message",
            "content": {"type": "input_image", "image_url": "{{item.image_url}}", "detail": "auto"},
        },
    ],
}

data_source = build_model_target_data_source(
    items=eval_items,
    model=IMAGE_MODEL,
    input_messages=input_messages,
    max_completion_tokens=100,
)

In [5]:
client = FoundryEvaluationClient(PROJECT_ENDPOINT)
try:
    evaluation = client.create_evaluation(
        name="Image Caption Evaluation - SDK 2.x",
        data_source_config=data_source_config,
        testing_criteria=[score_model_grader],
    )
    evaluation_run = client.create_run(
        evaluation_id=evaluation.id,
        name="Image Caption Evaluation - East US 2",
        data_source=data_source,
        metadata={"demo": "image", "region": "eastus2", "sdk": "azure-ai-projects-2.7.0"},
    )
    completed_run = client.wait_for_run(
        evaluation_id=evaluation.id,
        run_id=evaluation_run.id,
    )
    output_items = client.list_output_items(
        evaluation_id=evaluation.id,
        run_id=evaluation_run.id,
    )
finally:
    client.close()

Evaluation created: eval_df9d204f6d3a48219dfd35614126f6d5


Evaluation run created: evalrun_a4aff73e34884c59b3d12a242f7dd15b


Evaluation run evalrun_a4aff73e34884c59b3d12a242f7dd15b: in_progress


Evaluation run evalrun_a4aff73e34884c59b3d12a242f7dd15b: completed


## Results

Display grader status/score, reference caption, and the Foundry target's generated caption.

In [6]:
import pandas as pd

rows = []
for output_item in output_items:
    results = output_item.get("results") or []
    result = results[0] if results else {}
    datasource_item = output_item.get("datasource_item") or {}
    sample = output_item.get("sample") or {}
    sample_output = sample.get("output") or []
    rows.append({
        "id": output_item.get("id"),
        "status": result.get("status"),
        "score": result.get("score"),
        "reference_caption": datasource_item.get("caption"),
        "model_response": sample.get("output_text") or (sample_output[0].get("content") if sample_output else None),
    })

results_df = pd.DataFrame(rows)
display(results_df)
print(json.dumps({
    "evaluation_id": evaluation.id,
    "run_id": completed_run.id,
    "status": completed_run.status,
    "report_url": completed_run.report_url,
    "result_counts": completed_run.result_counts.model_dump() if hasattr(completed_run.result_counts, "model_dump") else completed_run.result_counts,
    "output_item_count": len(output_items),
}, indent=2))

,id,status,score,reference_caption,model_response
0,1,completed,1.0,"A view of a calm ocean with distant islands, f...","""Where lush forests meet serene seas."""
1,2,completed,1.0,"A workspace featuring a laptop, smartphone, bo...","""Workspace essentials: creativity meets techno..."
2,3,completed,0.9,A white mug featuring a design inspired by a C...,"""Coffee served with a touch of Cuban history."""
3,4,completed,1.0,Close-up of a rabbit's nose and whiskers.,"A close-up of a soft, furry nose belonging to ..."
4,5,completed,1.0,A bird spreads its wings on a rope near a boat...,A cormorant dries its wings while perched on a...
5,6,completed,1.0,A top-down view of a workspace with a computer...,A neatly organized workspace featuring a compu...
6,7,completed,1.0,A foggy road lined with tall trees.,"A serene, foggy road lined with tall trees cre..."
7,8,completed,1.0,A black-and-white close-up of pine cones on wood.,"""Nature's intricate beauty captured in monochr..."
8,9,completed,0.9,A row of glass jars placed on wooden stakes in...,"""Translucent glass jars perched on rustic wood..."
9,10,completed,0.7,A hazy beach scene with people near the water.,A serene beach scene with people enjoying the ...


{
  "evaluation_id": "eval_df9d204f6d3a48219dfd35614126f6d5",
  "run_id": "evalrun_a4aff73e34884c59b3d12a242f7dd15b",
  "status": "completed",
  "report_url": "https://ai.azure.com/nextgen/r/unl599BAScmvGnQUQCv2Ig,prakharg-demo-2026,,eastus2-prakharg-demo-2026,eastus2-prakharg-demo-2026/build/evaluations/eval_df9d204f6d3a48219dfd35614126f6d5/run/evalrun_a4aff73e34884c59b3d12a242f7dd15b",
  "result_counts": {
    "errored": 0,
    "failed": 0,
    "passed": 10,
    "total": 10,
    "skipped": 0
  },
  "output_item_count": 10
}
